# Multi-Regional AeroMAPS Scenarios - Integrated Feature

This notebook demonstrates the integrated multi-regional feature in AeroMAPS.

## Overview

The multi-regional mode allows you to:
1. Run multiple regional AeroMAPS scenarios in parallel
2. Automatically apply GEMSEO namespaces to isolate regional I/O
3. Aggregate regional outputs into global metrics using configurable rules

## Configuration

Multi-regional mode is configured via a single YAML file (`regionalisation.yaml`) that contains:
- Standard AeroMAPS configuration (models, etc.)
- A `regionalisation` section specifying regions and aggregation rules

## 1. Prepare Regional Data

First, we generate the partitioning data for each region.

In [ ]:
from aeromaps.utils.functions import create_partitioning

# Generate partitioning data for both regions
create_partitioning(file="data/region_france/aeroscope_data.csv", path="data/region_france")
create_partitioning(file="data/region_germany/aeroscope_data.csv", path="data/region_germany")
print("✓ Partitioning data generated for both regions")

## 2. Create Multi-Regional Process

The multi-regional process is created using the standard `create_process()` function.
The presence of a `regionalisation` section in the config file automatically triggers multi-regional mode.

In [ ]:
%matplotlib widget
from aeromaps import create_process

# Create the multi-regional process
# The regionalization.yaml file serves as both the main config and regionalisation config
process = create_process(configuration_file="regionalisation.yaml")

print(f"\n✓ Multi-regional process created")
print(f"  Regions: {process.list_regions()}")
print(f"  Total disciplines: {len(process.disciplines)}")

## 3. Execute Multi-Regional Analysis

The `compute()` method runs all regional scenarios in parallel using GEMSEO's MDAChain with MDAJacobi.

In [ ]:
import time

# Execute the multi-regional analysis
start_time = time.time()
process.compute()
elapsed_time = time.time() - start_time
print(f"✓ Computation complete in {elapsed_time:.2f} seconds")

## 4. Access Regional Outputs

You can access outputs for specific regions or all regions using the dedicated methods.

In [ ]:
import pandas as pd

# Get outputs for specific regions
fr_outputs = process.get_regional_outputs("FR")
de_outputs = process.get_regional_outputs("DE")

# Get global aggregated outputs
global_outputs = process.get_global_outputs()

print("Available regional variables (sample):")
print(f"  FR: {list(fr_outputs.keys())[:5]}...")
print(f"  DE: {list(de_outputs.keys())[:5]}...")
print(f"\nAvailable global variables:")
print(f"  {list(global_outputs.keys())}")

## 5. Compare Regional and Global Results

Let's compare CO2 emissions across regions and verify aggregation.

In [ ]:
import numpy as np

years = [2020, 2030, 2040, 2050]

print("=" * 60)
print("MULTI-REGIONAL RESULTS")
print("=" * 60)

# CO2 Emissions comparison
print("\nCO2 EMISSIONS (Mt):")
print("-" * 40)

data = {"Year": years}

if "co2_emissions" in fr_outputs:
    data["France"] = [fr_outputs["co2_emissions"][y] / 1e9 for y in years]
if "co2_emissions" in de_outputs:
    data["Germany"] = [de_outputs["co2_emissions"][y] / 1e9 for y in years]
if "co2_emissions" in global_outputs:
    data["GLOBAL"] = [global_outputs["co2_emissions"][y] / 1e9 for y in years]

df = pd.DataFrame(data).set_index("Year")
display(df)

In [ ]:
# Verify aggregation correctness
print("\nVERIFICATION - CO2 emissions 2050:")
print("-" * 40)

fr_val = fr_outputs.get("co2_emissions_including_energy", pd.Series())[2050] if "co2_emissions_including_energy" in fr_outputs else 0
de_val = de_outputs.get("co2_emissions_including_energy", pd.Series())[2050] if "co2_emissions_including_energy" in de_outputs else 0
global_val = global_outputs.get("co2_emissions_including_energy", pd.Series())[2050] if "co2_emissions_including_energy" in global_outputs else 0

print(f"  FR + DE = {(fr_val + de_val):.2f} Mt")
print(f"  Global  = {global_val:.2f} Mt")
print(f"  Match: {np.isclose(fr_val + de_val, global_val)}")

## 6. Access Raw MDAChain Data

You can also access the raw namespaced data from the MDAChain.

In [ ]:
# Access the raw local_data from the MDAChain
local_data = process.mda_chain.local_data

# Show some namespaced keys
namespaced_keys = [k for k in local_data.keys() if ":" in k]
print(f"Total namespaced variables: {len(namespaced_keys)}")
print(f"\nSample namespaced keys:")
for key in sorted(namespaced_keys)[:10]:
    print(f"  {key}")

## 7. Summary

The integrated multi-regional feature provides:

1. **Simplified API**: Use the standard `create_process()` with a config file
2. **Automatic namespacing**: Regional I/O variables are automatically prefixed
3. **Parallel execution**: Uses GEMSEO's MDAJacobi for parallel discipline execution
4. **Configurable aggregation**: Define sum or weighted average rules in YAML
5. **Easy data access**: `get_regional_outputs()` and `get_global_outputs()` methods

### Configuration Files Structure

```yaml
# config_multi_regional.yaml
regionalisation:
  regionalisation_file: "regionalisation.yaml"

# regionalisation.yaml
global_namespace: "overall"
regions:
  FR:
    config_file: "data/region_france/config.yaml"
  DE:
    config_file: "data/region_germany/config.yaml"
aggregation:
  sum:
    - co2_emissions
    - rpk
  weighted_average:
    - variable: load_factor
      weight_by: rpk
```

In [ ]:
# Cleanup
from aeromaps.utils.functions import clean_notebooks_on_tests

clean_notebooks_on_tests(globals())